<a href="https://colab.research.google.com/github/t3m14/neuro_huinya/blob/main/network_train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [43]:
import tensorflow as tf
from keras.layers import Dense
from keras.layers import Input, Embedding
from keras.layers import Flatten, concatenate, Concatenate, Dropout
from keras.layers import BatchNormalization
from keras.layers import Activation, LeakyReLU
from keras.optimizers import Adam
from keras.models import Model, load_model
from keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from keras.losses import mean_absolute_error as keras_MAE
from keras.utils import get_custom_objects
from keras.layers import Activation, LeakyReLU
from keras.models import load_model

In [44]:
import os
import pandas as pd

from matplotlib import pyplot as plt
import numpy as np

from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error
from sklearn.model_selection import KFold

from IPython.display import Image

In [45]:
DATA_FOLDER = '../data/processed_data'
START_DATA_FOLDER = '../data/house-pricing-prediction'
TARGET_COL_NAME = 'Price'
MAX_EMBEDDING_SIZE_FOR_CAT_FEATURE = 5
PICTURES_FOLDER = '../pictures/'

TRAIN_DATA_PATH = os.path.join(DATA_FOLDER, 'houses_train_data.csv')
TEST_DATA_PATH = os.path.join(DATA_FOLDER, 'houses_test_data.csv')

In [46]:
train_df = pd.read_csv(TRAIN_DATA_PATH).drop(columns=['Unnamed: 0'])
test_df = pd.read_csv(TEST_DATA_PATH).drop(columns=['Unnamed: 0'])

FileNotFoundError: [Errno 2] No such file or directory: '../data/processed_data/houses_train_data.csv'

Будем предсказывать цену в млн рублей, чтобы выход нейронной сети не предсказывал очень большие значения. Это может быть причиной нестабильного обучения

In [ ]:
train_df[TARGET_COL_NAME] = train_df[TARGET_COL_NAME] / 10**6

Приведем данные в удобный для модели формат

In [ ]:
cat_features = train_df.select_dtypes(include='int64').columns
X = train_df.drop(columns=[TARGET_COL_NAME])
y = train_df[TARGET_COL_NAME]

In [ ]:
X_cat = X.select_dtypes(include='int64')
X_ohe = X.select_dtypes(exclude='int64').values
y = y.values

Xt = test_df.values
Xt_cat = test_df.select_dtypes(include='int64')
Xt_cat = [Xt_cat[col_name].values for col_name in Xt_cat.columns]
Xt_ohe = test_df.select_dtypes(exclude='int64').values

In [ ]:
categorical_counts = {
    col_name: X_cat[col_name].nunique()
    for col_name in X_cat.columns
}

In [ ]:
X_cat = X_cat.values

### Обучим нейронную сеть

Добавим нужные функции потерь и функцию активации GeLU

In [ ]:
def gelu(x):
    return 0.5 * x * (1 + tf.tanh(tf.sqrt(2 / np.pi) * (x + 0.044715 * tf.pow(x, 3))))


def MAE(y_true, y_pred):
    try:
        return tf.py_function(mean_absolute_error, (y_true, y_pred), tf.double)
    except:
        return tf.py_func(mean_absolute_error, (y_true, y_pred), tf.double)


def MAPE(y_true, y_pred):
    try:
        return tf.py_function(mean_absolute_percentage_error, (y_true, y_pred), tf.double)
    except:
        return tf.py_func(mean_absolute_percentage_error, (y_true, y_pred), tf.double)


get_custom_objects().update({'gelu': Activation(gelu)})
get_custom_objects().update({'leaky-relu': Activation(LeakyReLU(alpha=0.2))})
get_custom_objects().update({'MAE': MAE})
get_custom_objects().update({'MAPE': MAPE})

Напишем пару вспомогательных функций для компиляции модели и отрисовки графиков

In [ ]:
def compile_model(model, loss, metrics, optimizer):
    model.compile(loss=loss, metrics=metrics, optimizer=optimizer)
    return model


def plot_keras_history(history, measures):
    """
    history: Keras training history
    measures = list of names of measures
    """
    rows = len(measures) // 2 + len(measures) % 2
    fig, panels = plt.subplots(rows, 2, figsize=(15, 5))
    plt.subplots_adjust(top = 0.99, bottom=0.01, hspace=0.4, wspace=0.2)
    try:
        panels = [item for sublist in panels for item in sublist]
    except:
        pass
    for k, measure in enumerate(measures):
        panel = panels[k]
        panel.set_title(measure + ' history')
        panel.plot(history.epoch, history.history[measure], label="Train "+measure)
        panel.plot(history.epoch, history.history["val_"+measure], label="Validation "+measure)
        panel.set(xlabel='epochs', ylabel=measure)
        panel.legend()

    plt.show(fig)

In [ ]:
def tabular_dnn(numeric_variables, categorical_variables, categorical_counts,
                first_dense = 256, second_dense = 256, dense_dropout = 0.2,
                activation_type=gelu):

    # Обработка числовых фичей
    numerical_inputs = Input(shape=(numeric_variables,))
    numerical_feature_selection = numerical_inputs

    # Обработка категориальных фичей через эмбединг слой
    categorical_inputs = []
    categorical_embeddings = []
    for category in categorical_variables:
        categorical_inputs.append(Input(shape=[1], name=category))
        category_counts = categorical_counts[category]
        categorical_embeddings.append(
            Embedding(category_counts + 1,
                      min(int(category_counts / 1.5 + 1), MAX_EMBEDDING_SIZE_FOR_CAT_FEATURE),
                      name = category + "_embed")(categorical_inputs[-1]))

    categorical_logits = Concatenate(name = "categorical_conc")([Flatten()(cat_emb)
                                                                 for cat_emb in categorical_embeddings])

    # Добавим пару полносвязных слоев, на которых производится предсказание
    x = concatenate([numerical_feature_selection, categorical_logits])
    x = BatchNormalization()(x)

    x = Dense(first_dense, activation=activation_type)(x)
    x = BatchNormalization()(x)
    x = Dropout(dense_dropout)(x)

    x = Dense(second_dense, activation=activation_type)(x)
    x = Dropout(dense_dropout)(x)
    x = BatchNormalization()(x)

    # Последний слой, который предсказывает цену (в млн)
    output = Dense(1, activation="relu")(x)

    model = Model([numerical_inputs] + categorical_inputs, output)

    return model

Напишем генератор батча (чтобы компоновать наши данные в батчи)

In [ ]:
def batch_generator(X_ohe, X_cat, y, cv=5, batch_size=64, random_state=None):
    '''
    Returns a batch from X, y
    random_state allows determinism
    different scikit-learn CV strategies are possible
    '''
    folds = len(y) // batch_size
    if isinstance(cv, int):
        kf = KFold(n_splits=cv,
                              shuffle=True,
                              random_state=random_state)
    else:
        kf = cv

    while True:
        for _, batch_index in kf.split(X_ohe, y):
            numeric_input = X_ohe[batch_index].astype(np.float32)
            categorical_input = [X_cat[i][batch_index] for i in range(len(X_cat))]
            target = y[batch_index]
            yield [numeric_input] + categorical_input, target

In [ ]:
# Global training settings
SEED = 42
FOLDS = 5
MAX_EPOCHS = 100
BATCH_SIZE = 16
LR = 0.0001

measure_to_monitor = 'val_MAE'
modality = 'min'

# Определим колбэки для обучения

early_stopping = EarlyStopping(monitor=measure_to_monitor,
                               mode=modality,
                               patience=5,
                               verbose=0)

model_checkpoint = ModelCheckpoint('best.model',
                                   monitor=measure_to_monitor,
                                   mode=modality,
                                   save_best_only=True,
                                   verbose=0)

model_reduce_lr = ReduceLROnPlateau(monitor=measure_to_monitor,
                                    mode=modality,
                                    factor=0.5,
                                    patience=2,
                                    min_lr=1e-6,
                                    verbose=1)
# Определим конфиг для модели
model_params = {
    "numeric_variables" : X_ohe.shape[1],
    "categorical_variables" : categorical_counts.keys(),
    "categorical_counts" : categorical_counts,
    "feature_selection_dropout" : 0.0,
    "categorical_dropout" : 0.3,
    "first_dense" : 64,
    "second_dense" : 64,
    "dense_dropout" : 0.3,
    "activation_type" : 'relu'
}

И, наконец, запустим обучение модели

In [ ]:
# Setting the CV strategy
skf = KFold(n_splits=FOLDS,
                      shuffle=True,
                      random_state=SEED)

# CV Iteration: we store best epochs, oof and cv testv prediciton
mae_list = list()
mape_list = list()
oof = np.zeros(len(X))
cv_test_preds = np.zeros(len(Xt))

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
    # Соберем модель
    model = compile_model(tabular_dnn(**model_params),
                          keras_MAE,
                          [MAE, MAPE],
                          Adam(learning_rate=LR))

    # Создадим учебный и валидационный датасеты для фолдов
    X_cv_ohe = X_ohe[train_idx].astype(np.float32)
    X_cv_cat = X_cat[train_idx]
    X_cv_cat = [X_cv_cat[:, i] for i in range(X_cv_cat.shape[1])]

    y_cv = y[train_idx]
    X_oof_ohe = X_ohe[test_idx].astype(np.float32)
    X_oof_cat = X_cat[test_idx]
    X_oof_cat = [X_oof_cat[:, i] for i in range(X_oof_cat.shape[1])]

    y_oof = y[test_idx]

    # Зададим генераторы батчей для тренировки и валидации
    train_batch = batch_generator(X_cv_ohe,
                                  X_cv_cat,
                                  y_cv,
                                  batch_size=BATCH_SIZE,
                                  random_state=SEED)
    val_batch = batch_generator(X_oof_ohe,
                                X_oof_cat,
                                y_oof,
                                batch_size=BATCH_SIZE,
                                random_state=SEED)

    train_steps = len(y_cv) // BATCH_SIZE
    validation_steps = len(y_oof) // BATCH_SIZE

    # Обучение
    history = model.fit_generator(train_batch,
                                  validation_data=val_batch,
                                  epochs=MAX_EPOCHS,
                                  steps_per_epoch=train_steps,
                                  validation_steps=validation_steps,
                                  callbacks=[model_checkpoint, early_stopping, model_reduce_lr],
                                  verbose=1)

    # Отчет по обучению (графички)
    print("\nFOLD %i" % fold)
    plot_keras_history(history, measures=['MAPE', 'MAE', 'loss'])

    # Финальное предсказание на валидации и подсчет метрик

    model = load_model('best.model')

    preds = model.predict([X_oof_ohe] + X_oof_cat,
                          verbose=1,
                          batch_size=BATCH_SIZE).flatten()

    oof[test_idx] = preds

    mae_list.append(mean_absolute_error(y_oof, preds))

    mape_list.append(mean_absolute_percentage_error(y_oof, preds))

    # Предсказание на тесте (усредненное будет отправлено на Kaggle)
    cv_test_preds += model.predict([Xt_ohe] + Xt_cat,
                                   verbose=1,
                                   batch_size=BATCH_SIZE).flatten() / FOLDS


Посмотрим на метрики после cv (с высокой долей уверенности результаты будут не хуже, чем приведенные значения метрик), т.к. здесь учтена зависимость от данных

In [ ]:
np.mean(mae_list) + np.std(mae_list)

In [ ]:
np.mean(mape_list) + np.std(mape_list)

### Отправим результаты на проверку на Kaggle

Вспомним, что месяц с номером 1 был в единственном экземпляре как в трейне, так и в тесте. Поэтому в тесте мы сделаем немного "нечестно" и предскажем значение цены как среднее среди всех предсказаний.

In [ ]:
SUBMISSIONS_FOLDER = '../data/submissions/'

In [ ]:
test_df = pd.read_csv('../data/house-pricing-prediction/houses_test_data.csv')
test_df['Date'] = pd.to_datetime(test_df['Date'], dayfirst=True)

assert (test_df['Date'].dt.month == 1).sum() == 1
month_equal_1_index = list(test_df.index[test_df['Date'].dt.month == 1])[0]



In [ ]:
month_equal_1_index

In [ ]:
df = pd.read_csv(
    os.path.join(SUBMISSIONS_FOLDER, 'sample_submission.csv')
)
df.loc[df.index != month_equal_1_index, TARGET_COL_NAME] = cv_test_preds * 10**6
df.loc[df.index == month_equal_1_index, TARGET_COL_NAME] = cv_test_preds.mean() * 10**6
df.to_csv(os.path.join(SUBMISSIONS_FOLDER, 'first_by_5_folds_nn_submission.csv'), index=False)

Данное решение показывает следующий результат на Kaggle (*2 место на Public и Private*), причем на Private скор даже лучше, чем на Public.

In [ ]:
Image(filename=os.path.join(PICTURES_FOLDER, 'first_results_picture.png'))